In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import networkx as nx
import json
# Import the LangChain Neo4jGraph wrapper
from langchain_community.graphs import Neo4jGraph

In [ ]:
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "12345678"
NEO4J_DATABASE = "small"

In [ ]:
# --- 3. Function to Load from Neo4j (using LangChain) to NetworkX ---
def load_langchain_neo4j_to_networkx(uri, user, password):
    """
    Connects to Neo4j using LangChain's Neo4jGraph, fetches the graph data,
    and returns a NetworkX MultiDiGraph.
    """
    # Using MultiDiGraph because Neo4j allows multiple relationships (of potentially
    # different types) between the same two nodes and directed relationships.
    G = nx.MultiDiGraph()

    try:
        # Instantiate the LangChain Neo4jGraph wrapper
        print("Initializing LangChain Neo4jGraph...")
        graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE)
        # Note: Neo4jGraph doesn't have a specific 'verify_connectivity' like the driver,
        # but errors usually occur during the first query if connection fails.

        # === Define Cypher Queries (same as before) ===
        nodes_query = """
            MATCH (n)
            RETURN id(n) AS neo4j_id, labels(n) AS labels, properties(n) AS properties
        """
        rels_query = """
            MATCH (start)-[r]->(end)
            RETURN id(start) AS start_id, id(end) AS end_id, type(r) AS type, properties(r) AS properties
        """

        # === Fetch Nodes using graph.query ===
        print("Fetching nodes from Neo4j via LangChain wrapper...")
        # graph.query returns a list of dictionaries
        nodes_result = graph.query(nodes_query)
        nodes_added = 0
        for record in nodes_result:
            node_id = record["neo4j_id"]
            labels = record["labels"]
            properties = record["properties"]
            # Add node with its properties and store labels in an attribute
            G.add_node(node_id, labels=labels, **properties)
            nodes_added += 1
        print(f"Added {nodes_added} nodes to NetworkX graph.")

        # === Fetch Relationships using graph.query ===
        print("Fetching relationships from Neo4j via LangChain wrapper...")
        rels_result = graph.query(rels_query)

        edges_added = 0
        for record in rels_result:
            start_id = record["start_id"]
            end_id = record["end_id"]
            rel_type = record["type"]
            properties = record["properties"]
            # Add edge. Use rel_type as the key for MultiDiGraph distinction.
            # Also store the type as an edge attribute for easy access.
            G.add_edge(start_id, end_id, key=rel_type, type=rel_type, **properties)
            edges_added += 1
        print(f"Added {edges_added} edges to NetworkX graph.")

    except Exception as e:
        # Catch potential errors during connection or query execution
        print(f"Error interacting with Neo4j via LangChain wrapper: {e}")
        # Return an empty graph or raise the exception depending on desired behavior
        return nx.MultiDiGraph()
    # Note: Neo4jGraph manages its own driver instance, so explicit closing isn't typically done here.

    print(f"\nNetworkX graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G

# --- 4. Load the graph from Neo4j using the LangChain wrapper ---
nx_graph = load_langchain_neo4j_to_networkx(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)

In [ ]:
nx_graph

In [ ]:
path = "small_dump.gml"

In [ ]:
nx.write_gml(nx_graph, path)

In [ ]:
loaded_graph = nx.read_gml(path)

In [ ]:
for node_id, attributes in loaded_graph.nodes(data=True):
    print(f"Node: {node_id}, Attributes: {attributes}")
    break